In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import sys
import os

# Get the current notebook directory and move up one level
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [34]:
from dotenv import load_dotenv
import os
load_dotenv()
from src.pr_reviewer import PRReviewer

In [35]:

github_token=os.getenv("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("Github_token not found")

In [36]:
#intialize the reviewer
reviewer=PRReviewer(github_token=github_token)

In [37]:
repo_url="https://github.com/Bhavesh-girase/pr-reviewer-test.git"
owner="Bhavesh-girase"
repository="pr-reviewer-test"
pull_number=1

In [38]:
github_files = reviewer.github.get_pull_request_files(
    owner,
    repository,
    pull_number,
)

github_files

[{'sha': '54c36dd5a3b4b14288fe4e29f02ba7ab441bc571',
  'filename': 'auth.py',
  'status': 'modified',
  'additions': 2,
  'deletions': 1,
  'changes': 3,
  'blob_url': 'https://github.com/Bhavesh-girase/pr-reviewer-test/blob/405d837365ad6feb357987f43b1a523b41508e85/auth.py',
  'raw_url': 'https://github.com/Bhavesh-girase/pr-reviewer-test/raw/405d837365ad6feb357987f43b1a523b41508e85/auth.py',
  'contents_url': 'https://api.github.com/repos/Bhavesh-girase/pr-reviewer-test/contents/auth.py?ref=405d837365ad6feb357987f43b1a523b41508e85',
  'patch': '@@ -2,5 +2,6 @@\n \n \n def verify_token(token):\n-    payload = jwt.decode(token)\n+    payload = jwt.decode(token,\n+    options={"verify_signature":False})\n     return payload\n\\ No newline at end of file'},
 {'sha': '1a39dfbe6ddb8193841039d0d415467c3f3547a8',
  'filename': 'database.py',
  'status': 'modified',
  'additions': 1,
  'deletions': 2,
  'changes': 3,
  'blob_url': 'https://github.com/Bhavesh-girase/pr-reviewer-test/blob/405d83

In [39]:
print(github_files[0]['patch'])

@@ -2,5 +2,6 @@
 
 
 def verify_token(token):
-    payload = jwt.decode(token)
+    payload = jwt.decode(token,
+    options={"verify_signature":False})
     return payload
\ No newline at end of file


In [40]:
# running the complete pipeline
reviews=reviewer.review(
    repo_url=repo_url,
    owner=owner,
    repository=repository,
    pull_number=pull_number,
)

Updating pr-reviewer-test...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 1844.34it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
print(reviews[0]['review'])

summary='The change introduces a critical security vulnerability by disabling JWT signature verification.' issues=[ReviewIssue(severity=<Severity.HIGH: 'HIGH'>, category=<Category.SECURITY: 'Security'>, title='JWT Signature Verification Disabled', description='The `jwt.decode` function is called with `options={"verify_signature":False}`. This explicitly disables the cryptographic signature verification of the JSON Web Token. This means that the application will accept any token as valid, even if it has been tampered with or completely forged by an attacker, as long as its structure is correct. This completely undermines the security guarantees of JWTs and allows for unauthorized access or privilege escalation.', suggestion='Signature verification is fundamental to the security of JWTs. It must be re-enabled. Ensure a strong secret key is used and passed to the `jwt.decode` function, along with the expected algorithms. For example: `payload = jwt.decode(token, YOUR_SECRET_KEY, algorithm

In [58]:

from langchain_chroma import Chroma

from src.config import embedding_function
from src.ingestion.data_utils import get_chroma_directory

repo_name = "chat"

vectorstore = Chroma(
    persist_directory=str(
        get_chroma_directory(repo_name)
    ),
    embedding_function=embedding_function,
)

collection = vectorstore._collection

In [57]:
results=collection.delete(where={"relative_path":"ChatContext.jsx"})



In [54]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])


In [55]:
print(results)

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [56]:
results["ids"]

[]